# StateSet Agents v1.0 — River AI 0.11 Full-Lifecycle Training, 6-Domain Sandbox RL & Serving (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/river_ai_full_lifecycle_colab.ipynb)

> **Zero GPUs required in Colab — River AI `0.11` runs remote autograd, RL, and hosted checkpoint serving!**
>
> **Important Python version note:** `river-client 0.11` requires **Python ≥ 3.12**. Because some Google Colab runtimes still default to Python 3.11, Cell 1 automatically creates a fast isolated **Python 3.12 `uv` environment** whenever `sys.version_info < (3, 12)` so every River `0.11` command works out of the box!
>
> Add your **`RIVER_API_KEY`** under **Colab Secrets** (left sidebar key icon) before running live cells. Without a key, all offline contract + 6-domain `RolloutEngine` cells still pass deterministically!

## 1. Bootstrap Python 3.12 Runtime & Install `stateset-agents[river]`

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

INSTALL_SPEC = os.environ.get(
    "STATESET_RIVER_INSTALL_SPEC",
    "stateset-agents[river] @ git+https://github.com/stateset/stateset-agents.git@release/0.99.1",
)

if sys.version_info >= (3, 12):
    RIVER_PY = sys.executable
    subprocess.check_call([RIVER_PY, "-m", "pip", "install", "-q", INSTALL_SPEC])
else:
    print(f"Colab kernel is Python {sys.version_info.major}.{sys.version_info.minor}; bootstrapping Python 3.12 via uv...")
    if shutil.which("uv") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    venv_dir = Path("/content/.river-py312") if Path("/content").exists() else Path(".river-py312")
    subprocess.check_call(["uv", "venv", "--python", "3.12", str(venv_dir)])
    RIVER_PY = str(venv_dir / "bin" / "python")
    subprocess.check_call(["uv", "pip", "install", "--python", RIVER_PY, INSTALL_SPEC])

out = subprocess.check_output(
    [RIVER_PY, "-c", "import stateset_agents, river_client; print(stateset_agents.__version__, river_client.__version__)"],
    text=True,
)
print("Ready (stateset-agents, river-client):", out.strip())

## 2. Load `RIVER_API_KEY` from Colab Secrets & Run `river-preflight`

In [ ]:
import json
import os
import subprocess

api_key = os.environ.get("RIVER_API_KEY", "").strip()
if not api_key:
    try:
        from google.colab import userdata
        api_key = (userdata.get("RIVER_API_KEY") or "").strip()
    except Exception:
        api_key = ""

env = dict(os.environ)
if api_key:
    env["RIVER_API_KEY"] = api_key

BASE_MODEL = "Qwen/Qwen3.6-35B-A3B-FP8"
cmd = [RIVER_PY, "-m", "stateset_agents.cli", "river-preflight"]
if api_key:
    cmd.extend(["--live", "--base-model", BASE_MODEL])

proc = subprocess.run(cmd, env=env, capture_output=True, text=True, check=False)
print(proc.stdout or proc.stderr)

## 3. Run All 6 Enterprise Verifiable Sandboxes Inside Real `river_client.rl.RolloutEngine`

Verifies `river_mcp_sandbox_factory(domain)` across `commerce`, `coding`, `sql`, `rest_api`, `docker_code`, and `web_agent` inside the real `river-client 0.11` `RolloutEngine` (zero API spend).

In [ ]:
import subprocess

script = '''
import asyncio
from stateset_agents.remote.river_environment import river_mcp_sandbox_factory

DOMAINS = ["commerce", "coding", "sql", "rest_api", "docker_code", "web_agent"]
for domain in DOMAINS:
    factory = river_mcp_sandbox_factory(domain)
    env = factory()
    obs = env.reset()
    print(f"  [OK] domain={domain:12s} obs_chars={len(str(obs))}")
'''
subprocess.check_call([RIVER_PY, "-c", script])

## 4. Generate a Reproducible River AI `0.11` RL Blueprint (`cispo` / `dro`)

In [ ]:
import subprocess

cmd = [
    RIVER_PY, "-m", "stateset_agents.cli", "rl",
    "--cloud", "river",
    "--algorithm", "cispo",
    "--model", BASE_MODEL,
    "--json",
]
print(subprocess.check_output(cmd, text=True))

## 5. Live River AI `0.11` Managed SFT, Checkpoint Chat & Dedicated Deployment Lifecycle (Opt-In)

> **Cost safety:** Set `RUN_LIVE_RIVER_SFT = True` and/or `RUN_LIVE_DEPLOYMENT_SMOKE = True` only when you want to incur live River usage. Dedicated deployments always execute inside `try / finally: manager.delete_deployment(...)` so idle replicas never keep billing.

In [ ]:
import json
import subprocess
from pathlib import Path

RUN_LIVE_RIVER_SFT = False
RUN_LIVE_DEPLOYMENT_SMOKE = False

sample_path = Path("river_colab_train.jsonl")
sample_rows = [
    {
        "messages": [
            {"role": "user", "content": "Order ORD-77701 arrived damaged."},
            {"role": "assistant", "content": "I have initiated a full refund for damaged order ORD-77701 and emailed your confirmation."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Cancel subscription sub_101 immediately."},
            {"role": "assistant", "content": "Subscription sub_101 is now canceled with no further recurring charges."},
        ]
    },
]
sample_path.write_text("\n".join(json.dumps(r) for r in sample_rows) + "\n", encoding="utf-8")

if RUN_LIVE_RIVER_SFT and api_key:
    subprocess.check_call(
        [
            RIVER_PY, "-m", "stateset_agents.cli", "train-remote",
            "--provider", "river",
            "--dataset", str(sample_path),
            "--base-model", BASE_MODEL,
            "--output-dir", "outputs/river_colab_sft",
            "--num-epochs", "1",
            "--lora-r", "16",
        ],
        env=env,
    )
    print(
        subprocess.check_output(
            [
                RIVER_PY, "-m", "stateset_agents.cli", "river-chat",
                "--checkpoint", "outputs/river_colab_sft",
                "--prompt", "Order ORD-88802 arrived damaged.",
            ],
            env=env,
            text=True,
        )
    )
else:
    print("Dry preview only (set RUN_LIVE_RIVER_SFT=True with RIVER_API_KEY to train live on River).")

if RUN_LIVE_DEPLOYMENT_SMOKE and api_key and Path("outputs/river_colab_sft/river_checkpoint.json").exists():
    deploy_script = '''
import os
from stateset_agents.remote.river_serving import RiverDeploymentManager

mgr = RiverDeploymentManager(api_key=os.environ["RIVER_API_KEY"])
dep_id = None
try:
    dep = mgr.create_deployment(checkpoint_path="outputs/river_colab_sft", unified_replicas=1)
    dep_id = dep["id"]
    print("Created deployment:", dep_id)
    ready = mgr.wait_for_deployment(dep_id, timeout_seconds=300)
    print("Status:", ready.get("status"))
    print("Usage:", mgr.get_deployment_usage(dep_id))
finally:
    if dep_id:
        mgr.delete_deployment(dep_id)
        print("Deleted deployment cleanly:", dep_id)
    mgr.close()
'''
    subprocess.check_call([RIVER_PY, "-c", deploy_script], env=env)